In [15]:
from models.ViT.vit import ViT
from models.CNN.simple_cnn import SimpleCNN
from data.preprocessing import preprocess 
from data.image_dataset_train_test_split import image_dataset_train_test_split
from data.image_dataset import ImageDataset
from training.loop import fit, predict

from pathlib import Path
import torch

In [11]:
preprocess(
    src_dir=Path("../data/raw"),
    dst_dir=Path("../data/processed"),
    img_size=224
)

/home/dell/miniconda3/envs/ri/lib/python3.8/site-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/home/dell/miniconda3/envs/ri/lib/python3.8/site-packages/PIL/Image.py:3368: DecompressionBombWarning: Image size (96714256 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Academic_Art: 1305 images found -> 1305 saved
Art_Nouveau: 3035 images found -> 3035 saved
Baroque: 5312 images found -> 5312 saved
Expressionism: 2607 images found -> 2607 saved
Japanese_Art: 2235 images found -> 2235 saved
Neoclassicism: 3115 images found -> 3115 saved
Primitivism: 1324 images found -> 1324 saved
Realism: 5373 images found -> 5373 saved
Renaissance: 6192 images found -> 6192 saved
Rococo: 2521 images found -> 2521 saved
Romanticism: 6813 images found -> 6813 saved
Symbolism: 1510 images found -> 1510 saved
Western_Medieval: 1158 images found -> 1158 saved


{'Academic_Art': 1305,
 'Art_Nouveau': 3035,
 'Baroque': 5312,
 'Expressionism': 2607,
 'Japanese_Art': 2235,
 'Neoclassicism': 3115,
 'Primitivism': 1324,
 'Realism': 5373,
 'Renaissance': 6192,
 'Rococo': 2521,
 'Romanticism': 6813,
 'Symbolism': 1510,
 'Western_Medieval': 1158}

In [12]:
image_dataset_train_test_split(src_path="../data/processed", dest_path="../data/training", test_size=0.2)

Academic_Art: 1305 found -> 1044 train, 261 test
Art_Nouveau: 3035 found -> 2428 train, 607 test
Baroque: 5312 found -> 4250 train, 1062 test
Expressionism: 2607 found -> 2086 train, 521 test
Japanese_Art: 2235 found -> 1788 train, 447 test
Neoclassicism: 3115 found -> 2492 train, 623 test
Primitivism: 1324 found -> 1060 train, 264 test
Realism: 5373 found -> 4299 train, 1074 test
Renaissance: 6192 found -> 4954 train, 1238 test
Rococo: 2521 found -> 2017 train, 504 test
Romanticism: 6813 found -> 5451 train, 1362 test
Symbolism: 1510 found -> 1208 train, 302 test
Western_Medieval: 1158 found -> 927 train, 231 test


{'Academic_Art': {'train': 1044, 'test': 261},
 'Art_Nouveau': {'train': 2428, 'test': 607},
 'Baroque': {'train': 4250, 'test': 1062},
 'Expressionism': {'train': 2086, 'test': 521},
 'Japanese_Art': {'train': 1788, 'test': 447},
 'Neoclassicism': {'train': 2492, 'test': 623},
 'Primitivism': {'train': 1060, 'test': 264},
 'Realism': {'train': 4299, 'test': 1074},
 'Renaissance': {'train': 4954, 'test': 1238},
 'Rococo': {'train': 2017, 'test': 504},
 'Romanticism': {'train': 5451, 'test': 1362},
 'Symbolism': {'train': 1208, 'test': 302},
 'Western_Medieval': {'train': 927, 'test': 231}}

In [39]:
dataset = ImageDataset("../data/training/train", max_per_class=1000)

print("Num samples:", len(dataset))     
print("Classes:", dataset.classes)      
print("Weights:", dataset.weights)

Num samples: 12927
Classes: ['Academic_Art', 'Art_Nouveau', 'Baroque', 'Expressionism', 'Japanese_Art', 'Neoclassicism', 'Primitivism', 'Realism', 'Renaissance', 'Rococo', 'Romanticism', 'Symbolism', 'Western_Medieval']
Weights: tensor([1.0056, 1.0056, 1.0056, 1.0056, 1.0056, 1.0056, 1.0056, 1.0056, 1.0056,
        1.0056, 1.0056, 1.0056, 0.9322])


In [40]:
test_dataset = ImageDataset("../data/training/test", max_per_class=400);

print("Num samples:", len(test_dataset))     
print("Classes:", test_dataset.classes)      
print("Weights:", test_dataset.weights)

Num samples: 3363
Classes: ['Academic_Art', 'Art_Nouveau', 'Baroque', 'Expressionism', 'Japanese_Art', 'Neoclassicism', 'Primitivism', 'Realism', 'Renaissance', 'Rococo', 'Romanticism', 'Symbolism', 'Western_Medieval']
Weights: tensor([1.0089, 1.0089, 1.0089, 1.0089, 1.0089, 1.0089, 1.0089, 1.0089, 1.0089,
        1.0089, 1.0089, 1.0089, 0.8930])


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = ViT(img_size=224, in_channels=3, num_classes=len(dataset.classes), embed_dim=128,
            heads=4, N=4, P=16)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = torch.nn.CrossEntropyLoss(weight=dataset.weights)

fit(model, dataset, optimizer, criterion, device=device, epochs=5, batch_size=16)

epoch [1/5]
train loss: 2.3615, train accuracy: 0.1970
validation loss: 2.2312, validation accuracy: 0.2615
epoch [2/5]
train loss: 2.1257, train accuracy: 0.2844
validation loss: 2.0675, validation accuracy: 0.3215
epoch [3/5]
train loss: 1.9710, train accuracy: 0.3415
validation loss: 1.9827, validation accuracy: 0.3586


In [29]:
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

In [ ]:
y_pred = predict(model, test_dataset, batch_size=16, device=device)

In [ ]:
y_test = torch.tensor([label for _, label in test_dataset.samples])

In [ ]:
y_test.shape

In [ ]:
y_pred.shape

In [ ]:
cm = confusion_matrix(y_test, y_pred)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
f1_score(y_test, y_pred, average='weighted')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=test_dataset.classes
).plot(
    xticks_rotation=90,
    cmap="Reds"
)

plt.tight_layout()
plt.show()

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

cnn_model = SimpleCNN(in_channels=3, num_of_classes=len(dataset.classes), img_size=224)

optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=3e-4)
criterion = torch.nn.CrossEntropyLoss(weight=dataset.weights)

fit(cnn_model, dataset, optimizer, criterion, device=device, epochs=5, batch_size=16)